## Add Demodata

In [1]:
from dotenv import load_dotenv
import chromadb
from chromadb.config import Settings
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

from datasets import load_dataset, Dataset
from datasets import get_dataset_config_names
from mylibs.classes.AppSettings import AppSettings
settings = AppSettings()

ModuleNotFoundError: No module named 'datasets'

### german Q/A data for testing from HuggingFace

In [ ]:
name_ds = "deepset/germanquad"
configs = get_dataset_config_names(name_ds)
print(configs)

demodata: Dataset = load_dataset(name_ds, split='test')  # type: ignore

print(demodata.num_rows)
print(demodata.shape) # Shape of the dataset (number of columns, number of rows).
print(demodata.column_names)
print(demodata.features)

### slice into pieces of 50 

In [ ]:
import json
# split in first 100 and rest
ab = 150
bis = 200 # dataset.num_rows

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

data = []
for i in range(bis-ab):
  question = {}
  question["id"] = f'Q_{slice["id"][i]}'
  # question["process_id"]= str(slice["id"][i])
  question["process_id"]= f"Slice_{ab}_{bis}"
  question["gdpr_id"]= str(slice["id"][i])
  question["topic"]= slice["question"][i]
  question["type"]= "question"
  question["len"] = len(slice["question"][i])
  question["document"]= slice["question"][i]
  data.append(question)

  answer = {}
  answer["id"] = f'A_{slice["id"][i]}'
  # answer["process_id"]= str(slice["id"][i])
  answer["process_id"]= f"Slice_{ab}_{bis}"
  answer["gdpr_id"]= str(slice["id"][i])
  answer["topic"]= slice["answers"][i]["text"][0]
  answer["type"]= "answer"
  answer["len"] = len(slice["context"][i])
  answer["document"]= slice["context"][i]
  data.append(answer)


## Or

In [ ]:
import json

n = 3

steps = 100

ab = n * steps
bis = (n+1) * steps

slice = demodata[ab:bis]
# !!! len(slice) is 4 because ['id', 'context', 'question', 'answers']

# slice = dataset[anz:]

datas = []
for i in range(bis-ab):
  data = {}
  data["id"] = slice["id"][i]
  data["process_id"]= f"Slice_{ab}_{bis}"
  data["gdpr_id"]= str(slice["id"][i])
  data["topic"]= slice["question"][i]
  data["type"]= "answer"

  document = f'Q: {slice["question"][i]}\nA: {slice["answers"][i]["text"][0]}\nTopic: {slice["context"][i]}'
  data["document"]= document
  data["len"] = len(document)

  datas.append(data)

In [ ]:
len(datas)
# data

In [ ]:
datas[0]["process_id"]

In [ ]:
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()
elastic_vector_search = ElasticsearchStore(
    index_name="test_index",
    embedding=embedding,
    es_cloud_id="the_cloud_id",
    es_user="elastic",
    es_password="your_Password",
    es_api_key="your_apikey",
)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./demodata.txt", encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()

In [ ]:
docs[0]


In [ ]:
elastic_vector_search.add_documents(docs)

In [ ]:
import os
import requests

api_url = "http://127.0.0.1:8000/ai/embedding/insert/"

headers = {
  'Content-Type': 'application/json',
  'access_token': settings.AI_API_KEY
}

response = requests.put(api_url, json=datas, headers=headers)
if response.status_code != 200:
  print("Error at: ", response.json)

In [ ]:
print(response.json())
len(response.json())

In [ ]:
import pandas as pd

df = pd.DataFrame(data)
print(df)

In [ ]:
from elasticsearch import Elasticsearch

client = Elasticsearch(
  "https://...",
  api_key=""
)

In [ ]:
# API key should have cluster monitor rights
client.info()